Project: /mediapipe/_project.yaml
Book: /mediapipe/_book.yaml

<link rel="stylesheet" href="/mediapipe/site.css">

# Hand gesture recognition model customization guide

<table align="left" class="buttons">
  <td>
    <a href="https://colab.research.google.com/github/googlesamples/mediapipe/blob/main/examples/customization/gesture_recognizer.ipynb" target="_blank">
      <img src="https://developers.google.com/static/mediapipe/solutions/customization/colab-logo-32px_1920.png" alt="Colab logo"> Run in Colab
    </a>
  </td>

  <td>
    <a href="https://github.com/googlesamples/mediapipe/blob/main/examples/customization/gesture_recognizer.ipynb" target="_blank">
      <img src="https://developers.google.com/static/mediapipe/solutions/customization/github-logo-32px_1920.png" alt="GitHub logo">
      View on GitHub
    </a>
  </td>
</table>

In [ ]:
#@title License information
# Copyright 2023 The MediaPipe Authors.
# Licensed under the Apache License, Version 2.0 (the "License");
#
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

The MediaPipe Model Maker package is a low-code solution for customizing on-device machine learning (ML) Models.

This notebook shows the end-to-end process of customizing a gesture recognizer model for recognizing some common hand gestures in the [HaGRID](https://www.kaggle.com/datasets/innominate817/hagrid-sample-30k-384p) dataset.

## Prerequisites

Install the MediaPipe Model Maker package.

In [ ]:
!pip install --upgrade pip
!pip install mediapipe-model-maker
#connect to 2025.7 runtime

Import the required libraries.

In [ ]:
from google.colab import files
import os
import tensorflow as tf
assert tf.__version__.startswith('2')

from mediapipe_model_maker import gesture_recognizer

import matplotlib.pyplot as plt

## Simple End-to-End Example

This end-to-end example uses Model Maker to customize a model for on-device gesture recognition.

### Get the dataset

The dataset for gesture recognition in model maker requires the following format: `<dataset_path>/<label_name>/<img_name>.*`. In addition, one of the label names (`label_names`) must be `none`. The `none` label represents any gesture that isn't classified as one of the other gestures.

This example uses a rock paper scissors dataset sample which is downloaded from GCS.

In [ ]:
!wget https://huggingface.co/datasets/GestureDetectionConnoisseurs/hagrid_subsets/resolve/main/hagrid-export_100_images.zip -P /content/hagrid_dataset


In [ ]:
!unzip /content/hagrid_dataset/hagrid-export_100_images.zip -d /content/hagrid_dataset
dataset_path = "/content/hagrid_dataset"

In [ ]:
dataset_path = "/content/hagrid_dataset"
print(dataset_path)
labels = []
for i in os.listdir(dataset_path):
  if os.path.isdir(os.path.join(dataset_path, i)):
    labels.append(i)
print(labels)

To better understand the dataset, plot a couple of example images for each gesture.

In [ ]:
NUM_EXAMPLES = 5

for label in labels:
  label_dir = os.path.join(dataset_path, label)
  example_filenames = os.listdir(label_dir)[:NUM_EXAMPLES]
  fig, axs = plt.subplots(1, NUM_EXAMPLES, figsize=(10,2))
  for i in range(NUM_EXAMPLES):
    axs[i].imshow(plt.imread(os.path.join(label_dir, example_filenames[i])))
    axs[i].get_xaxis().set_visible(False)
    axs[i].get_yaxis().set_visible(False)
  fig.suptitle(f'Showing {NUM_EXAMPLES} examples for {label}')

plt.show()

### Run the example
The workflow consists of 4 steps which have been separated into their own code blocks.

**Load the dataset**

Load the dataset located at `dataset_path` by using the `Dataset.from_folder` method. When loading the dataset, run the pre-packaged hand detection model from MediaPipe Hands to detect the hand landmarks from the images. Any images without detected hands are ommitted from the dataset. The resulting dataset will contain the extracted hand landmark positions from each image, rather than images themselves.

The `HandDataPreprocessingParams` class contains two configurable options for the data loading process:
* `shuffle`: A boolean controlling whether to shuffle the dataset. Defaults to true.
* `min_detection_confidence`: A float between 0 and 1 controlling the confidence threshold for hand detection.

Split the dataset: 80% for training, 10% for validation, and 10% for testing.

In [ ]:
!mv /content/hagrid_dataset/no_gesture/ /content/hagrid_dataset/none
data = gesture_recognizer.Dataset.from_folder(
    dirname=dataset_path,
    hparams=gesture_recognizer.HandDataPreprocessingParams()
)
train_data, rest_data = data.split(0.8)
validation_data, test_data = rest_data.split(0.5)

**Train the model**

Train the custom gesture recognizer by using the create method and passing in the training data, validation data, model options, and hyperparameters. For more information on model options and hyperparameters, see the [Hyperparameters](#hyperparameters) section below.

In [ ]:
hparams = gesture_recognizer.HParams(export_dir="exported_model")
options = gesture_recognizer.GestureRecognizerOptions(hparams=hparams)
model = gesture_recognizer.GestureRecognizer.create(
    train_data=train_data,
    validation_data=validation_data,
    options=options
)

**Evaluate the model performance**

After training the model, evaluate it on a test dataset and print the loss and accuracy metrics.

In [ ]:
loss, acc = model.evaluate(test_data, batch_size=1)
print(f"Test loss:{loss}, Test accuracy:{acc}")

**Export to Tensorflow Lite Model**

After creating the model, convert and export it to a Tensorflow Lite model format for later use on an on-device application. The export also includes model metadata, which includes the label file.

In [ ]:
model.export_model()
!ls exported_model
files.download('exported_model/gesture_recognizer.task')

In [ ]:
# Export the underlying Keras model as a SavedModel
tf.saved_model.save(
    model._model,
    'saved_model'
)

!ls saved_model

In [ ]:
print(type(train_data))
print([x for x in dir(train_data) if not x.startswith('__')])
for image, label in train_data.gen_tf_dataset().take(1):
    print("Image shape:", image.shape)
    print("Image dtype:", image.dtype)
    print("Label:", label)

In [ ]:
import tensorflow as tf
import numpy as np


def representative_data_gen():
    tf_dataset = train_data.gen_tf_dataset()
    for embedding, _ in tf_dataset.take(200):
        # Already in the right shape (1, 128) and dtype float32
        yield [embedding]

converter = tf.lite.TFLiteConverter.from_saved_model('./saved_model')

converter = tf.lite.TFLiteConverter.from_saved_model('./saved_model')
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.float32  # embeddings stay float32
converter.inference_output_type = tf.float32

quantized_tflite = converter.convert()

with open('gesture_recognizer_quantized.tflite', 'wb') as f:
    f.write(quantized_tflite)

In [ ]:
interpreter = tf.lite.Interpreter(
    model_path='gesture_recognizer_quantized.tflite'
)
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Use a sample embedding from your dataset
for sample, label in train_data.gen_tf_dataset().take(1):
    interpreter.set_tensor(input_details[0]['index'], sample)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])
    print("Predicted:", output)
    print("True label:", label.numpy())

**Generating .task file**

This section generates the .task file by replacing the original weights with the quantized weights of our specific model

In [ ]:
import zipfile

with zipfile.ZipFile('exported_model/gesture_recognizer.task', 'r') as z_outer:
    for inner_name in z_outer.namelist():
        inner_bytes = z_outer.read(inner_name)
        try:
            import io
            with zipfile.ZipFile(io.BytesIO(inner_bytes), 'r') as z_inner:
                print(f"{inner_name} contains:", z_inner.namelist())
        except:
            print(f"{inner_name} is not a zip, size: {len(inner_bytes)} bytes")

In [ ]:
import zipfile
import shutil
import io

with open('gesture_recognizer_quantized.tflite', 'rb') as f:
    quantized_bytes = f.read()

with zipfile.ZipFile('exported_model/gesture_recognizer.task', 'r') as z_outer:
    outer_files = {}
    for outer_name in z_outer.namelist():
        inner_bytes = z_outer.read(outer_name)

        if outer_name == 'hand_gesture_recognizer.task':
            # Rebuild this inner bundle with the quantized classifier swapped in
            inner_buffer = io.BytesIO()
            with zipfile.ZipFile(io.BytesIO(inner_bytes), 'r') as z_inner:
                with zipfile.ZipFile(inner_buffer, 'w') as z_inner_out:
                    for inner_name in z_inner.namelist():
                        if inner_name == 'custom_gesture_classifier.tflite':
                            z_inner_out.writestr(inner_name, quantized_bytes)
                        else:
                            z_inner_out.writestr(inner_name, z_inner.read(inner_name))
            outer_files[outer_name] = inner_buffer.getvalue()
        else:
            outer_files[outer_name] = inner_bytes

# Write the final .task bundle
with zipfile.ZipFile('gesture_recognizer_quantized.task', 'w') as z_out:
    for name, data in outer_files.items():
        z_out.writestr(name, data)

print("Done! gesture_recognizer_quantized.task is ready.")

### Print Statistics
This section will print the statistics of the new quantized model. Specifically, comparing file size of the .task file of the original v.s. the quantized model, filze size comparing the models that got quantized, and the final test accuracy of the quantized model.

In [ ]:
import os

original_size = os.path.getsize('exported_model/gesture_recognizer.task')
quantized_size = os.path.getsize('gesture_recognizer_quantized.task')
reduction = (1 - quantized_size / original_size) * 100

print(f"Original:  {original_size / 1024:.1f} KB")
print(f"Quantized: {quantized_size / 1024:.1f} KB")
print(f"Reduction: {reduction:.1f}%")

In [ ]:
import zipfile
import io

def get_inner_file_size(task_path, inner_task, inner_tflite):
    with zipfile.ZipFile(task_path, 'r') as z_outer:
        inner_bytes = z_outer.read(inner_task)
        with zipfile.ZipFile(io.BytesIO(inner_bytes), 'r') as z_inner:
            return z_inner.getinfo(inner_tflite).file_size

orig = get_inner_file_size(
    'exported_model/gesture_recognizer.task',
    'hand_gesture_recognizer.task',
    'custom_gesture_classifier.tflite'
)
quant = get_inner_file_size(
    'gesture_recognizer_quantized.task',
    'hand_gesture_recognizer.task',
    'custom_gesture_classifier.tflite'
)

print(f"Original classifier:  {orig / 1024:.1f} KB")
print(f"Quantized classifier: {quant / 1024:.1f} KB")
print(f"Reduction: {((1 - quant/orig) * 100):.1f}%")

In [ ]:
import tensorflow as tf
import numpy as np

interpreter = tf.lite.Interpreter(model_path='gesture_recognizer_quantized.tflite')
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

correct = 0
total = 0
label_names = test_data.label_names
per_class_correct = {name: 0 for name in label_names}
per_class_total   = {name: 0 for name in label_names}

for embedding, label in test_data.gen_tf_dataset():
    interpreter.set_tensor(input_details[0]['index'], embedding)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])

    predicted_idx = np.argmax(output[0])
    true_idx      = np.argmax(label[0])
    true_name     = label_names[true_idx]

    per_class_total[true_name] += 1
    total += 1

    if predicted_idx == true_idx:
        correct += 1
        per_class_correct[true_name] += 1

print(f"\nOverall accuracy: {correct}/{total} ({100 * correct / total:.1f}%)\n")
print(f"{'Gesture':<25} {'Correct':<10} {'Total':<10} {'Accuracy'}")
print("-" * 55)
for name in label_names:
    t = per_class_total[name]
    c = per_class_correct[name]
    acc = (100 * c / t) if t > 0 else 0
    print(f"{name:<25} {c:<10} {t:<10} {acc:.1f}%")